In [1]:
#Import essential libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import concat_ws, col, isnan, isnull, when, count, round, date_format, countDistinct, max

#Initiate Spark Session
spark = SparkSession.builder \
    .appName("Batch_Data_Engineering_Pipeline") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.3") \
    .getOrCreate()

spark

In [2]:
#Read and load 1st CSV file. Use seperator and inferSchema to recognize data types 
subscribers_df = spark.read.csv("../01.Data/subscribers.csv", sep = ',', inferSchema=True)

In [3]:
subscribers_df.show(5)

+------+----------+
|   _c0|       _c1|
+------+----------+
|123456|2021-08-10|
|123451|2021-08-10|
|123452|2021-08-10|
|123453|2021-08-10|
|123454|2021-08-10|
+------+----------+
only showing top 5 rows



In [4]:
#Check data types. Date is ok but id neeeds to be converted to string
subscribers_df.dtypes

[('_c0', 'int'), ('_c1', 'date')]

In [5]:
#Check initial database lines
subscribers_df.count()

79

In [6]:
#Initial database has no headers. Column context and names are given in the instructions
subscribers_df = subscribers_df.toDF("subscriber_id", "activation_date")
subscribers_df.show(5)

+-------------+---------------+
|subscriber_id|activation_date|
+-------------+---------------+
|       123456|     2021-08-10|
|       123451|     2021-08-10|
|       123452|     2021-08-10|
|       123453|     2021-08-10|
|       123454|     2021-08-10|
+-------------+---------------+
only showing top 5 rows



In [7]:
#Correct the data types format before we begin sanity & correctness tests
subscribers_df = subscribers_df.withColumn("subscriber_id", col("subscriber_id").cast("int"))
subscribers_df.dtypes

[('subscriber_id', 'int'), ('activation_date', 'date')]

In [8]:
#Check for null values, in this case there are none
subscribers_df.select([count(when(col(c).isNull(), c)).alias(c) for c in subscribers_df.columns]).show()

+-------------+---------------+
|subscriber_id|activation_date|
+-------------+---------------+
|            0|              0|
+-------------+---------------+



In [9]:
#Check if ids are unique. In this case we have duplicates (77 unique versus 79 entries in total)
subscribers_df.select(countDistinct("subscriber_id")).show()
subscribers_df.count()

+-----------------------------+
|count(DISTINCT subscriber_id)|
+-----------------------------+
|                           77|
+-----------------------------+



79

In [10]:
#Filter duplicate id entries in a separate table and bring the activation dates with an inner join. We notice that duplicate ids have different activation dates
duplicate_ids = subscribers_df.groupBy("subscriber_id") \
    .count() \
    .filter("count > 1") \
    .select("subscriber_id")
subscribers_df.join(duplicate_ids, on="subscriber_id", how="inner") \
    .orderBy("subscriber_id") \
    .show()

+-------------+---------------+
|subscriber_id|activation_date|
+-------------+---------------+
|       123456|     2021-08-10|
|       123456|     2020-12-22|
|       345991|     2021-07-09|
|       345991|     2020-12-22|
+-------------+---------------+



In [11]:
#Take the decision for duplicate entries to keep only the entries with the latest activation date (this should be a management decision)
#If the decision was to keep same ids with different activation date we could perform a duplicates check on the concatenated column (id&activation date). In this case no duplicate entries would have been found.
subscribers_unique_df = subscribers_df.groupBy("subscriber_id").agg(max("activation_date").alias("activation_date"))

In [12]:
#New count validates the two removed entries
subscribers_unique_df.count()

77

In [13]:
#Create the subscribers final table that includes the requested concatenate column and has the requested data-type formats.
subscribers_final_df = subscribers_unique_df \
    .withColumn("row_key", concat_ws("_", 
                                     col("subscriber_id").cast("string"),
                                     date_format("activation_date", "yyyyMMdd"))) \
    .withColumn("sub_id", col("subscriber_id").cast("string")) \
    .withColumn("act_dt", col("activation_date").cast("date")) \
    .select("row_key", "sub_id", "act_dt")

subscribers_final_df.show(5)

+---------------+------+----------+
|        row_key|sub_id|    act_dt|
+---------------+------+----------+
|345998_20210709|345998|2021-07-09|
|123455_20210810|123455|2021-08-10|
|345994_20210709|345994|2021-07-09|
|543221_20201222|543221|2020-12-22|
|876549_20201222|876549|2020-12-22|
+---------------+------+----------+
only showing top 5 rows



In [14]:
#Rechecking the data types just to be 100% sure
subscribers_final_df.dtypes

[('row_key', 'string'), ('sub_id', 'string'), ('act_dt', 'date')]

In [15]:
#Save the final table in the postgres database
subscribers_final_df.write \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://container_postgres:5432/postgres_db") \
    .option("dbtable", "subscribers") \
    .option("user", "user_user") \
    .option("password", "pass_pass") \
    .option("driver", "org.postgresql.Driver") \
    .mode("overwrite") \
    .save()

In [16]:
#Review final table using a sample
subscribers_final_df.show(5)

+---------------+------+----------+
|        row_key|sub_id|    act_dt|
+---------------+------+----------+
|345998_20210709|345998|2021-07-09|
|123455_20210810|123455|2021-08-10|
|345994_20210709|345994|2021-07-09|
|543221_20201222|543221|2020-12-22|
|876549_20201222|876549|2020-12-22|
+---------------+------+----------+
only showing top 5 rows

